# Astro Collision Lab · circular-speed experiments
Use a CPU runtime and select **Runtime → Run all**. Three batches explore ten disk quadrature settings each. The objective is fewer force evaluations with at most 0.5% peak-normalized error against the repository initializer. A third disk thickness is reserved for the final check. Each batch uses up to ten numerical workers sharing one Colab runtime.

In [ ]:
import itertools, json, random, subprocess, sys, tempfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd

revision = "a2fb5f86d4eb3ce6ab6788441c5774bbca56e59c"
repo = Path(tempfile.mkdtemp()) / "astro"
subprocess.run(["git", "clone", "-q", "https://github.com/trdillman/astro-collision-lab.git", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "-q", revision], check=True)
sys.path.insert(0, str(repo / "packages/astro-initial-conditions/src"))
from astro_ics import disk_galaxy, galaxy_models as gm

baseline = (160, 128)
tolerance = 0.005
training = (0.2, 0.4)
holdout = 0.3
workers = 10
seed = 42

In [ ]:
def reference(h):
    group = disk_galaxy("bench", thickness=h)
    return gm.circular_curve(group[0], group, 1., .03, np)

def candidate(params, h, grid):
    rings, angles = params
    disk, halo = disk_galaxy("bench", thickness=h)
    edges = np.linspace(0, disk.radius, rings + 1)
    radii = (edges[1:] + edges[:-1])[:, None] / 2
    mass = np.diff(gm.enclosed(disk, edges, np))[:, None]
    phi = (np.arange(angles) + .5) * 2 * np.pi / angles
    x, y = radii * np.cos(phi), radii * np.sin(phi)
    ratio = halo.radius / halo.scale_length
    norm = ratio**3 / (1 + ratio**2)**1.5
    force = halo.mass / norm * grid / (grid**2 + halo.scale_length**2 + .03**2)**1.5
    for i, r in enumerate(grid):
        dx = r - x
        force[i] += np.sum(mass * dx / (dx**2 + y**2 + .03**2 + (disk.thickness*.35)**2)**1.5) / angles
    return np.maximum(grid * force, 0.)

def score(params, h, ref):
    grid, target = ref
    values = candidate(params, h, grid)
    if not np.isfinite(values).all():
        raise ValueError("Nonfinite curve")
    return float(np.max(np.abs(values - target)) / np.max(target))

refs = {h: reference(h) for h in training}
for h, ref in refs.items():
    np.testing.assert_allclose(candidate(baseline, h, ref[0]), ref[1], rtol=1e-11, atol=1e-13)

In [ ]:
def evaluate(params):
    try:
        error = max(score(params, h, refs[h]) for h in training)
        return dict(params=params, error=error, passed=error <= tolerance)
    except Exception as exc:
        return dict(params=params, passed=False, failure=repr(exc))

rng = random.Random(seed)
remaining = set(itertools.product([20,32,48,64,80,96,128,160], [16,32,48,64,80,96,112,128])) - {baseline}
best = baseline
rows = []
with ThreadPoolExecutor(max_workers=workers) as pool:
    for epoch in range(1, 4):
        neighbors = sorted(p for p in remaining if p[0] == best[0] or p[1] == best[1])
        batch = rng.sample(neighbors, min(5, len(neighbors)))
        batch += rng.sample(sorted(remaining.difference(batch)), 10-len(batch))
        remaining.difference_update(batch)
        trials = list(pool.map(evaluate, batch))
        rows.extend(dict(t, epoch=epoch, parent=best) for t in trials)
        best = min([best] + [t["params"] for t in trials if t["passed"]], key=lambda p: p[0]*p[1])
        print(f"Round {epoch}: best {best}; {sum(t['passed'] for t in trials)}/10 within tolerance")
display(pd.DataFrame(rows))

In [ ]:
held_error = score(best, holdout, reference(holdout))
report = dict(revision=revision, numpy=np.__version__, seed=seed, workers=workers,
              training=training, holdout=holdout, baseline=baseline, selected=best,
              tolerance=tolerance, held_error=held_error, held_pass=held_error <= tolerance,
              work_reduction=1-best[0]*best[1]/(baseline[0]*baseline[1]),
              failures=sum("failure" in row for row in rows))
Path("curve-results.json").write_text(json.dumps(dict(summary=report, trials=rows), indent=2))
print(json.dumps(report, indent=2))
print("Download curve-results.json from Colab's Files panel.")